# ERA5-Land daily wind speed and direction, 2026 : Kakamega, Kilifi, Nyeri

Daily 10 m wind from ERA5-Land hourly reanalysis (~11 km), clipped per county and gap-filled.

**Outputs** (under `OUTPUT_DIR = F:\WIND_2026`):
- `county_daily_direct/<County>/` - raw Earth Engine daily export (4 bands, nodata -9999)
- `clipped_daily/<County>/` - re-clipped with rasterio
- `filled_daily/<County>/WIND_<County>_YYYY_MM_DD.tif` - **final gap-filled daily rasters**; bands: `wind_speed` (m/s, mean of the 24 hourly speeds), `u_mean`, `v_mean`, `wind_dir` (degrees, meteorological "from": 0 = N, 90 = E)
- `stats_daily/<County>_wind_daily_raw.csv` and `<County>_wind_daily_FILLED.csv`; `logs/` with gap-fill and failed-download logs

**Notes**
- Set to **October 2026 only** (`START_DATE`/`END_DATE`). Earlier months already on disk are left alone, and `<County>_wind_daily_FILLED.csv` is merged (October rows replaced, other months kept). The gap-fill log is written per run: `gap_fill_log_wind_<County>_20261001.csv`.
- The end date is capped at the latest day with all 24 hourly ERA5-Land images, so "up to now" means up to that day. Re-run any time to pick up new days; existing files are skipped.
- Direction fix: the numpy direction formulas (daily stats and the temporal gap-fill) used `arctan2(v, u)`, which mirrors the direction. They now use `180 + deg(arctan2(u, v))`. The run asserts both the Earth Engine and numpy conventions against N/E/S/W test vectors before downloading anything.
- If you already have `filled_daily` rasters or stats from the earlier version, re-run with `FORCE_REBUILD_COUNTY_OUTPUTS = True` once, or at least regenerate the filled files and CSVs: the old `wind_dir` values in gap-filled days and in the CSVs are affected. Pixels taken straight from Earth Engine were computed by the EE formula, which your check validates.
- Days download in parallel (`MAX_WORKERS`) with a progress bar. If the bar shows as text, install ipywidgets and restart the kernel.

In [ ]:
# Run once, then restart the kernel if pip installed anything
%pip install geemap geedim geopandas scikit-learn tqdm ipywidgets rasterio scipy matplotlib


In [ ]:
# =============================================================================
# wind_kakamega_daily.py — daily 10 m wind speed & direction per county
# from ERA5-Land hourly reanalysis, gap-filled the same way as the CHIRPS
# rainfall pipeline (spatial nearest-fill, then temporal fill from adjacent
# real days) -- with special handling for wind_dir since direction is
# circular (mean(350, 10) is NOT 180, it's ~0).
#
# CHANGES FROM THE ORIGINAL
#   1. COUNTY (a list) -> COUNTIES (list) + run_county(county) (a string).
#      This fixes the os.path.join TypeError.
#   2. The date range is capped at the latest day ERA5-Land actually has,
#      so future dates are no longer "gap-filled" with copied data.
#   3. FORCE_REBUILD_COUNTY_OUTPUTS defaults to False so existing downloads
#      are reused. Set True only when you want a full re-download.
# =============================================================================

import os
import time
import bisect
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import geopandas as gpd
import ee
import geemap
import rasterio
from rasterio.mask import mask as rio_mask
from rasterio.features import geometry_mask
from scipy.ndimage import distance_transform_edt
import matplotlib.pyplot as plt

# ============================================================
# CONFIG
# ============================================================
EE_PROJECT = "lags-512907"
SHAPEFILE_PATH = r"C:\Users\locateIT1\Downloads\ken_admin1.shp"
NAME_COLUMN = "adm1_name"

# Counties to process (each gets its own folders, logs and stats CSVs).
COUNTIES = ["Kakamega", "Kilifi", "Nyeri"]

OUTPUT_DIR = r"F:\WIND_2026"
STATS_DIR = os.path.join(OUTPUT_DIR, "stats_daily")
LOG_DIR = os.path.join(OUTPUT_DIR, "logs")

START_DATE = "2026-01-01"
END_DATE = "2026-11-09"   # capped automatically at the latest COMPLETE ERA5-Land day available
MAX_WORKERS = 4           # days downloaded in parallel; lower to 2-3 if you see Earth Engine 429 / quota errors

ERA5_HOURLY = "ECMWF/ERA5_LAND/HOURLY"   # 0.1 deg (~11 km), land only
ERA5_SCALE = 11132                        # native pixel size in metres
NODATA_VALUE = -9999
RAD2DEG = 180.0 / np.pi

BAND_NAMES = ["wind_speed", "u_mean", "v_mean", "wind_dir"]  # export order

FORCE_REBUILD_COUNTY_OUTPUTS = False  # True = wipe raw/clipped outputs and re-download
ENABLE_GAP_FILLING = True
PLOT_SAMPLE_AFTER_RUN = True

for d in [STATS_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

# ============================================================
# EARTH ENGINE AUTH
# ============================================================
print("1. Initializing Earth Engine...")
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)


# ============================================================
# HELPERS
# ============================================================
_ALL_GDF = None


def load_county_gdf(county):
    """Read ken_admin1.shp once; return the rows whose NAME_COLUMN matches this county (EPSG:4326)."""
    global _ALL_GDF
    if _ALL_GDF is None:
        g = gpd.read_file(SHAPEFILE_PATH)
        g = g.set_crs(epsg=4326) if g.crs is None else g.to_crs(epsg=4326)
        g["_n"] = g[NAME_COLUMN].astype(str).str.title()
        _ALL_GDF = g
    gdf = _ALL_GDF[_ALL_GDF["_n"] == county.title()].drop(columns="_n").copy()
    if gdf.empty:
        raise ValueError(f"County {county!r} not found in column {NAME_COLUMN!r}; available: {sorted(_ALL_GDF['_n'].unique())}")
    if not gdf.geometry.is_valid.all():
        gdf["geometry"] = gdf.geometry.buffer(0)
    return gdf


def check_direction_convention():
    """Should print N->0.0, E->90.0, S->180.0, W->270.0"""
    pt = ee.Geometry.Point([37.0, -0.4])
    for u, v, label in [(0, -1, "N"), (-1, 0, "E"), (0, 1, "S"), (1, 0, "W")]:
        u_img = ee.Image.constant(u).rename("u")
        v_img = ee.Image.constant(v).rename("v")
        img = u_img.expression(
            "mod(180 + 57.29577951308232 * atan2(v, u), 360)",
            {"u": u_img, "v": v_img})
        val = img.reduceRegion(ee.Reducer.first(), pt, 1000).values().get(0).getInfo()
        print(f"  {label} -> {round(val, 1)}")
        _exp = {"N": 0, "E": 90, "S": 180, "W": 270}[label]
        assert abs(((val - _exp + 180) % 360) - 180) < 1.0, f"Earth Engine wind_dir convention is wrong for {label}: got {val}"


def latest_era5_date():
    """Date (normalized) of the most recent hourly ERA5-Land image."""
    last = (ee.ImageCollection(ERA5_HOURLY)
            .sort("system:time_start", False)
            .first()
            .get("system:time_start")
            .getInfo())
    ts = pd.Timestamp(last, unit="ms")
    day = ts.normalize()
    return day if ts.hour == 23 else day - pd.Timedelta(days=1)   # only fully complete days (24 hourly images)


def wind_day_image(aoi, year, month, day):
    """(image, n_hours). Bands, in this exact order: wind_speed (true mean
    of hourly instantaneous speeds, m/s), u_mean, v_mean, wind_dir (deg,
    meteorological 'direction from', via atan2(v, u) -- same convention as
    check_direction_convention())."""
    start = ee.Date.fromYMD(year, month, day)
    end = start.advance(1, "day")

    coll = (ee.ImageCollection(ERA5_HOURLY)
            .filterDate(start, end)
            .filterBounds(aoi)
            .select(["u_component_of_wind_10m", "v_component_of_wind_10m"]))

    n_hours = coll.size().getInfo()
    if n_hours == 0:
        return None, 0

    speed = coll.map(lambda im: im.expression(
        "sqrt(u*u + v*v)",
        {"u": im.select("u_component_of_wind_10m"),
         "v": im.select("v_component_of_wind_10m")}
    ).rename("wind_speed")).mean()

    uv = coll.mean().rename(["u_mean", "v_mean"])

    direction = uv.expression(
        "mod(180 + 57.29577951308232 * atan2(v, u), 360)",
        {"u": uv.select("u_mean"), "v": uv.select("v_mean")}
    ).rename("wind_dir")

    return speed.addBands(uv).addBands(direction), n_hours


def clip_and_unmask(img, geom):
    return img.clip(geom).unmask(NODATA_VALUE, sameFootprint=False)


def set_nodata_tag(path, nodata_value):
    with rasterio.open(path, "r+") as dst:
        dst.nodata = nodata_value  # applies to every band


def clip_raster(raster_path, gdf, out_path):
    with rasterio.open(raster_path) as src:
        geom = gdf.to_crs(src.crs) if gdf.crs != src.crs else gdf
        geoms = [feat["geometry"] for feat in geom.__geo_interface__["features"]]
        out_image, out_transform = rio_mask(src, geoms, crop=True, nodata=NODATA_VALUE)
        out_meta = src.meta.copy()
        out_meta.update(height=out_image.shape[1], width=out_image.shape[2],
                        transform=out_transform, nodata=NODATA_VALUE)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with rasterio.open(out_path, "w", **out_meta) as dst:
        dst.write(out_image)
    return out_image


def nearest_fill(data, invalid_mask, aoi_mask):
    fillable = invalid_mask & aoi_mask
    if not fillable.any():
        return data, np.zeros_like(fillable)
    valid = aoi_mask & ~invalid_mask
    if not valid.any():
        return data, np.zeros_like(fillable)
    idx = distance_transform_edt(~valid, return_distances=False, return_indices=True)
    filled = data.copy()
    filled[fillable] = data[tuple(ind[fillable] for ind in idx)]
    return filled, fillable


# ============================================================
# MAIN PIPELINE FOR ONE COUNTY
# ============================================================
def run_county(county, date_range):
    print(f"\n{'=' * 60}\nCOUNTY: {county}\n{'=' * 60}")

    county_tif_dir = os.path.join(OUTPUT_DIR, "county_daily_direct", county)   # raw EE export
    county_clip_dir = os.path.join(OUTPUT_DIR, "clipped_daily", county)        # rasterio re-clip
    county_filled_dir = os.path.join(OUTPUT_DIR, "filled_daily", county)       # gap-filled output
    gap_fill_log = os.path.join(LOG_DIR, f"gap_fill_log_wind_{county}_{date_range[0]:%Y%m%d}.csv")

    if FORCE_REBUILD_COUNTY_OUTPUTS:
        for base_dir in [county_tif_dir, county_clip_dir, county_filled_dir]:
            if os.path.isdir(base_dir):
                shutil.rmtree(base_dir)
                print(f"Wiped {base_dir} for a clean rebuild.")
    for d in [county_tif_dir, county_clip_dir, county_filled_dir]:
        os.makedirs(d, exist_ok=True)

    # ---------- Load geometry ----------
    print("2. Loading boundary...")
    county_gdf = load_county_gdf(county)
    try:
        county_geom = county_gdf.union_all()
    except AttributeError:  # older geopandas
        county_geom = county_gdf.unary_union
    aoi = ee.Geometry(county_geom.__geo_interface__)

    # ---------- Download ----------
    print("4. Downloading daily wind rasters...")
    failed_downloads = []

    def fetch_one_day(d):
        year, month, day = d.year, d.month, d.day
        out_path = os.path.join(county_tif_dir, f"WIND_{county}_{year}_{month:02d}_{day:02d}.tif")
        if os.path.exists(out_path):
            return
        try:
            img, n_hours = wind_day_image(aoi, year, month, day)
            if img is None:
                raise ValueError("No ERA5-Land hourly data found for this date")
            if n_hours < 24:
                raise ValueError(f"only {n_hours}/24 hourly images available")
            clipped = clip_and_unmask(img, aoi).set("year", year, "month", month, "day", day)
            for k in range(1, 4):
                try:
                    geemap.ee_export_image(
                        clipped, filename=out_path, scale=ERA5_SCALE, region=aoi,
                        crs="EPSG:4326", file_per_band=False,
                    )
                    break
                except Exception:
                    if k == 3:
                        raise
                    time.sleep(3 * k)
            set_nodata_tag(out_path, NODATA_VALUE)
        except Exception as e:
            tqdm.write(f"   Failed {county} {year}-{month:02d}-{day:02d}: {e}")
            failed_downloads.append({"date": d.date(), "error": str(e)})

    # days are downloaded in parallel; the progress bar (ipywidgets) advances as each day finishes
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = [pool.submit(fetch_one_day, d) for d in date_range]
        for f in tqdm(as_completed(futures), total=len(futures), desc=f"{county} download", unit="day"):
            try:
                f.result()
            except Exception as e:
                tqdm.write(f"   Unexpected error: {e}")

    if failed_downloads:
        fail_log = os.path.join(LOG_DIR, f"failed_wind_downloads_{county}.csv")
        pd.DataFrame(failed_downloads).to_csv(fail_log, index=False)
        print(f"   {len(failed_downloads)} dates failed -- see {fail_log}")

    # ---------- Rasterio clip + raw stats ----------
    print("5. Clipping and computing raw daily stats...")
    raw_records = []
    for fname in sorted(os.listdir(county_tif_dir)):
        if not fname.endswith(".tif"):
            continue
        parts = fname.replace(".tif", "").split("_")  # WIND, county, year, month, day
        year, month, day = int(parts[-3]), int(parts[-2]), int(parts[-1])
        raster_path = os.path.join(county_tif_dir, fname)
        out_path = os.path.join(county_clip_dir, fname)
        try:
            arr = clip_raster(raster_path, county_gdf, out_path)  # shape (4, H, W)
            speed, u, v = arr[0], arr[1], arr[2]
            valid = speed != NODATA_VALUE
            if valid.any():
                mean_speed = float(speed[valid].mean())
                mean_u, mean_v = float(u[valid].mean()), float(v[valid].mean())
                mean_dir = float((180 + RAD2DEG * np.arctan2(mean_u, mean_v)) % 360)
            else:
                mean_speed = mean_u = mean_v = mean_dir = np.nan
            raw_records.append({"date": pd.Timestamp(year, month, day),
                                "wind_speed_ms": mean_speed,
                                "u_mean_ms": mean_u, "v_mean_ms": mean_v,
                                "wind_dir_deg": mean_dir,
                                "valid_pixels": int(valid.sum())})
        except Exception as e:
            print(f"   Failed to clip {fname}: {e}")

    if raw_records:
        pd.DataFrame(raw_records).sort_values("date").to_csv(
            os.path.join(STATS_DIR, f"{county}_wind_daily_raw.csv"), index=False)

    # ---------- Gap filling ----------
    # wind_speed, u_mean, v_mean are linear -> nearest-neighbor spatially,
    # average-of-real-neighbor-days temporally.
    # wind_dir is CIRCULAR -> tier 1 (copy of one real pixel) is safe as-is;
    # tier 2 recomputes direction from the averaged u/v, never from averaged degrees.
    if ENABLE_GAP_FILLING:
        print("6. Gap-filling missing days and missing pixels...")

        def clip_path(dt):
            return os.path.join(county_clip_dir,
                                f"WIND_{county}_{dt.year}_{dt.month:02d}_{dt.day:02d}.tif")

        ref_path = next((clip_path(dt) for dt in date_range if os.path.exists(clip_path(dt))), None)
        if ref_path is None:
            print("   No clipped wind files found -- skipping gap-fill.")
        else:
            with rasterio.open(ref_path) as src:
                profile = src.profile.copy()
                transform = src.transform
                out_shape = (src.height, src.width)

            aoi_mask = ~geometry_mask(list(county_gdf.to_crs(profile["crs"]).geometry),
                                      out_shape=out_shape, transform=transform, invert=False)
            n_aoi = int(aoi_mask.sum())

            # Pass 1: tier-1 spatial fill, per band
            arrays = {}
            gap_fill_records = []
            n_missing_files = 0
            n_fully_nodata = 0

            for dt in date_range:
                path = clip_path(dt)
                if not os.path.exists(path):
                    n_missing_files += 1
                    continue
                with rasterio.open(path) as src:
                    raw = src.read().astype("float64")  # (4, H, W)
                    nodata = src.nodata if src.nodata is not None else NODATA_VALUE

                speed_invalid = (raw[0] == nodata) | np.isnan(raw[0])
                if (speed_invalid & aoi_mask).sum() == n_aoi:
                    n_fully_nodata += 1
                    continue  # whole day nodata -> tier 2

                day_bands = {}
                n_fixed_total = 0
                for bi, bname in enumerate(BAND_NAMES):
                    band = raw[bi]
                    invalid = (band == nodata) | np.isnan(band)
                    filled, fixed = nearest_fill(band, invalid, aoi_mask)
                    day_bands[bname] = filled
                    n_fixed_total += int(fixed.sum())
                if n_fixed_total:
                    gap_fill_records.append({"date": dt.date(), "tier": "spatial",
                                             "pixels_filled": n_fixed_total})
                arrays[dt] = day_bands

            n_real_days = len(arrays)
            print(f"   {n_real_days}/{len(date_range)} days had usable real data "
                  f"({n_missing_files} had no file, {n_fully_nodata} existed but were fully nodata)")

            # Pass 2: tier-2 temporal fill
            anchor_dates = sorted(arrays.keys())
            anchor_keys = [d.toordinal() for d in anchor_dates]
            n_unfillable = 0

            for dt in date_range:
                if dt in arrays:
                    continue
                pos = bisect.bisect_left(anchor_keys, dt.toordinal())
                prev_dt = anchor_dates[pos - 1] if pos > 0 else None
                next_dt = anchor_dates[pos] if pos < len(anchor_dates) else None

                if prev_dt is not None and next_dt is not None:
                    prev_b, next_b = arrays[prev_dt], arrays[next_dt]
                    speed_avg = np.nanmean(np.stack([prev_b["wind_speed"], next_b["wind_speed"]]), axis=0)
                    u_avg = np.nanmean(np.stack([prev_b["u_mean"], next_b["u_mean"]]), axis=0)
                    v_avg = np.nanmean(np.stack([prev_b["v_mean"], next_b["v_mean"]]), axis=0)
                    # Direction recomputed from the averaged vector, NOT averaged directly.
                    dir_avg = (180 + RAD2DEG * np.arctan2(u_avg, v_avg)) % 360
                    day_bands = {"wind_speed": speed_avg, "u_mean": u_avg,
                                 "v_mean": v_avg, "wind_dir": dir_avg}
                    source = f"avg({prev_dt.date()},{next_dt.date()})"
                elif prev_dt is not None:
                    day_bands, source = arrays[prev_dt], f"prev({prev_dt.date()})"
                elif next_dt is not None:
                    day_bands, source = arrays[next_dt], f"next({next_dt.date()})"
                else:
                    print(f"   WARNING: {dt.date()} has NO real day anywhere to fill from -- skipped.")
                    gap_fill_records.append({"date": dt.date(), "tier": "UNFILLABLE", "pixels_filled": 0})
                    n_unfillable += 1
                    continue

                gap_fill_records.append({"date": dt.date(), "tier": f"temporal:{source}",
                                         "pixels_filled": n_aoi})
                arrays[dt] = day_bands

            if n_unfillable:
                print(f"   {n_unfillable} day(s) could not be filled at all.")

            # Write every day (real, tier-1, tier-2) as a 4-band GeoTIFF
            out_profile = profile.copy()
            out_profile.update(nodata=NODATA_VALUE)
            for dt, day_bands in arrays.items():
                out_path = os.path.join(county_filled_dir,
                                        f"WIND_{county}_{dt.year}_{dt.month:02d}_{dt.day:02d}.tif")
                stack = np.stack([day_bands[b] for b in BAND_NAMES])
                stack[:, ~aoi_mask] = NODATA_VALUE
                stack[np.isnan(stack)] = NODATA_VALUE
                with rasterio.open(out_path, "w", **out_profile) as dst:
                    dst.write(stack.astype(out_profile["dtype"]))
                    dst.descriptions = tuple(BAND_NAMES)

            print(f"   Wrote {len(arrays)}/{len(date_range)} days to {county_filled_dir}")
            if gap_fill_records:
                pd.DataFrame(gap_fill_records).sort_values("date").to_csv(gap_fill_log, index=False)
                print(f"   Gap-fill log written to {gap_fill_log}")

            # Recompute daily stats from the filled output
            filled_records = []
            for dt, day_bands in arrays.items():
                speed_vals = day_bands["wind_speed"][aoi_mask]
                u_vals = day_bands["u_mean"][aoi_mask]
                v_vals = day_bands["v_mean"][aoi_mask]
                mean_speed = float(np.nanmean(speed_vals))
                mean_u, mean_v = float(np.nanmean(u_vals)), float(np.nanmean(v_vals))
                mean_dir = float((180 + RAD2DEG * np.arctan2(mean_u, mean_v)) % 360)
                filled_records.append({"date": dt, "wind_speed_ms": mean_speed,
                                       "u_mean_ms": mean_u, "v_mean_ms": mean_v,
                                       "wind_dir_deg": mean_dir})
            filled_df = pd.DataFrame(filled_records).sort_values("date").reset_index(drop=True)
            stats_out = os.path.join(STATS_DIR, f"{county}_wind_daily_FILLED.csv")
            if os.path.exists(stats_out):   # keep earlier months, replace only the days processed in this run
                old = pd.read_csv(stats_out, parse_dates=["date"])
                old = old[~old["date"].between(date_range[0], date_range[-1])]
                filled_df = pd.concat([old, filled_df], ignore_index=True).sort_values("date").reset_index(drop=True)
            filled_df.to_csv(stats_out, index=False)
            print(f"   Filled stats written to {stats_out} ({len(filled_df)} days in file, {len(arrays)} from this run)")

    # ---------- Visual QA ----------
    if PLOT_SAMPLE_AFTER_RUN:
        print("7. Plotting a sample wind_speed raster...")
        source_dir = county_filled_dir if ENABLE_GAP_FILLING else county_clip_dir
        sample_files = sorted(f for f in os.listdir(source_dir) if f.endswith(".tif")) \
            if os.path.isdir(source_dir) else []
        if sample_files:
            sample_path = os.path.join(source_dir, sample_files[0])
            with rasterio.open(sample_path) as src:
                speed = src.read(1).astype("float64")
                nodata = src.nodata if src.nodata is not None else NODATA_VALUE
            masked = np.ma.masked_equal(speed, nodata)
            cmap = plt.cm.viridis.copy()
            cmap.set_bad(color="none")
            fig, ax = plt.subplots(figsize=(6, 6))
            im = ax.imshow(masked, cmap=cmap)
            ax.axis("off")
            ax.set_title(f"{county} wind speed — {sample_files[0]}")
            fig.colorbar(im, ax=ax, label="Wind speed (m/s)", shrink=0.8)
            plt.tight_layout()
            plt.show()
        else:
            print("   No files found to plot.")


# ============================================================
# RUN
# ============================================================
print("3. Direction convention check:")
check_direction_convention()          # Earth Engine side (asserts N/E/S/W = 0/90/180/270)
for _u, _v, _exp in [(0, -1, 0), (-1, 0, 90), (0, 1, 180), (1, 0, 270)]:   # numpy side
    assert abs((((180 + RAD2DEG * np.arctan2(_u, _v)) % 360) - _exp + 180) % 360 - 180) < 1e-6
print("   numpy direction formula OK")

# Cap the date range at the latest day ERA5-Land has actually published
latest = latest_era5_date()
end = min(pd.Timestamp(END_DATE), latest)
date_range = pd.date_range(START_DATE, end, freq="D")
print(f"Latest ERA5-Land day available: {latest.date()}  ->  processing "
      f"{date_range[0].date()} to {date_range[-1].date()} ({len(date_range)} days)")

for county in COUNTIES:
    run_county(county, date_range)

print("\nDone. Daily wind pipeline complete.")